# 실험 01: OpenAI API 기본 호출 및 프롬프트 엔지니어링

## 1. 실험 제목과 목표
- **제목**: 프롬프트 및 파라미터 튜닝 비교 실험
- **목표**: System Prompt의 역할, Temperature에 따른 창의성 변화, 출력 형식 지정 방법을 비교 실험하고 실패 케이스를 통해 프롬프트 엔지니어링의 중요성을 학습합니다.

## 2. 실험 개요
1. **기본 예제**: System vs User 메시지의 차이
2. **비교 실험 1 (페르소나)**: 다양한 역할 부여에 따른 답변 스타일 변화
3. **비교 실험 2 (Temperature)**: 0.1, 0.7, 1.0 온도에 따른 출력 다양성
4. **비교 실험 3 (형식 지정)**: 자유 서술, Bullet, JSON 형식 유도
5. **실패 케이스**: 모호한 프롬프트와 과도한 제약 조건 시연

## 3. 라이브러리 import

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

## 4. 환경 변수 로드 및 client 생성

In [ ]:
# .env 파일에서 OPENAI_API_KEY 로드
load_dotenv()
client = OpenAI()

## 5. 기본 예제 (User 단독 vs System 추가)
System Prompt가 있을 때와 없을 때 모델의 기본 태도가 어떻게 다른지 확인합니다.

In [ ]:
question = "파이썬의 주요 특징을 한 문장으로 설명해줘."

# 1. User Message만 사용
res_user_only = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}]
)

# 2. System Message 추가
res_with_system = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 항상 답변 끝에 '삐리삐리, 로봇입니다.'를 붙여야 합니다."},
        {"role": "user", "content": question}
    ]
)

print("[User 단독] :", res_user_only.choices[0].message.content)
print("\n[System 추가] :", res_with_system.choices[0].message.content)

## 6. 비교 실험 1: System Prompt 페르소나 비교
동일한 질문에 대해 역할(Teacher, Senior Dev, Interviewer)에 따라 어조와 전문성이 어떻게 변하는지 비교합니다.

In [ ]:
question_persona = "객체지향 프로그래밍(OOP)이 뭐야?"
personas = {
    "선생님": "초등학생에게 코딩을 가르치는 다정한 선생님입니다. 아주 쉽고 비유를 들어 설명하세요.",
    "시니어 개발자": "20년 경력의 무뚝뚝한 시니어 개발자입니다. 전문 용어를 사용해 극도로 간결하게 핵심만 말하세요.",
    "면접관": "IT 기업 기술 면접관입니다. 개념을 묻고 난 뒤, 압박 질문을 하나 던지세요."
}

for role_name, system_prompt in personas.items():
    res_persona = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question_persona}
        ]
    )
    print(f"\n=== [{role_name} 역할] ===")
    print(res_persona.choices[0].message.content)

## 7. 비교 실험 2: Temperature 비교
`temperature` (0.0 ~ 2.0) 값에 따라 창의성과 일관성이 어떻게 달라지는지 확인합니다. (0에 가까울수록 정형적, 1 이상일수록 창의적/무작위적)

In [ ]:
question_temp = "새로운 카페 메뉴 이름을 2개만 지어줘."
temperatures = [0.1, 0.7, 1.2]  # 1.0 대신 1.2로 극적인 차이 유도

for temp in temperatures:
    res_temp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=temp,
        messages=[{"role": "user", "content": question_temp}]
    )
    print(f"\n=== [Temperature: {temp}] ===")
    print(res_temp.choices[0].message.content)

## 8. 비교 실험 3: 출력 형식 지정 (Output Format)
LLM 결과를 서비스에서 활용하기 위해 자유 서술형 대신 표, Bullet, JSON 등 특정 형식을 강제하는 실험입니다.

In [ ]:
target_info = "애플과 삼성의 대표 스마트폰 모델"

formats = {
    "자유 서술형": "자유롭게 문장으로 설명해줘.",
    "Bullet 형식": "Markdown bullet point 형식으로 간결하게 정리해줘.",
    "JSON 형식": "반드시 JSON 형식으로 응답해줘. 예시: {'Apple': '...', 'Samsung': '...'}"
}

for fmt_name, format_instruction in formats.items():
    res_fmt = client.chat.completions.create(
        model="gpt-4o-mini",
        # response_format={"type": "json_object"} 옵션을 주면 강제 JSON 모드가 되지만, 여기서는 Prompt로 유도해봅니다.
        messages=[
            {"role": "system", "content": format_instruction},
            {"role": "user", "content": target_info}
        ]
    )
    print(f"\n=== [{fmt_name}] ===")
    print(res_fmt.choices[0].message.content)

## 9. 실패/주의 케이스 시연
LLM이 어떻게 헷갈리거나 지시사항을 무시하는지 확인합니다.

In [ ]:
# 실패 1: 모호한 프롬프트
res_fail1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "그거 할 때 쓰는 거 추천해줘."}] # 맥락이 전혀 없음
)
print("[실패 1 - 모호한 질문의 결과]")
print(res_fail1.choices[0].message.content)
print("-> 결과: 모델이 '그거'가 무엇인지 모르기 때문에 매우 추상적이거나 환각적인 답변을 생성합니다.\n")

# 실패 2: 너무 많고 충돌하는 제약 조건
conflict_prompt = """
인공지능에 대해 설명해. 
조건 1: 반드시 3단어로만 답해. 
조건 2: 무조건 영어로 답해. 
조건 3: 시조(한국 전통 시) 형태로 답해. 
조건 4: 전문 용어를 10개 이상 포함해.
"""
res_fail2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": conflict_prompt}]
)
print("[실패 2 - 충돌하는 제약 조건의 결과]")
print(res_fail2.choices[0].message.content)
print("-> 결과: 3단어, 영어, 시조, 전문용어 10개가 서로 논리적으로 모순되므로, 모델이 일부 조건을 완전히 무시하게 됩니다.")

## 10. 결과 해석

1. **System Prompt**: 모델의 '메타 인지(페르소나, 규칙)'를 형성하는 가장 강력한 수단임을 확인했습니다.
2. **Temperature**: 목적(정보 제공 vs 아이디어 브레인스토밍)에 맞춰 온도를 조절하는 것이 중요합니다.
3. **출력 형식**: 애플리케이션(UI나 DB)과 연동하기 위해서는 JSON처럼 기계가 읽기 쉬운 구조화된 출력(Structured Output)이 필수적입니다.
4. **실패 케이스**: 프롬프트는 명확하고 구체적이어야 하며, 서로 충돌하는 조건을 피해야 양질의 답변을 얻을 수 있습니다.

## 11. Notion 실험 로그

### 발견한 문제점
* 프롬프트만으로 JSON을 유도했을 때, 가끔 마크다운 코드 블록(```json)이 포함되어 반환되어 파싱(Parsing) 에러를 유발할 위험이 있음을 확인함.
* 여러 개의 복잡한 지시사항이 섞이면 모델이 일부 지시사항(특히 문장 길이 제한 등)을 망각하거나 무시하는 현상 관찰됨.

### 다음 개선 방향
* Output Parser나 OpenAI의 `response_format`을 사용하여 100% 안정적인 JSON 추출(Structured Output)을 보장하는 방법 학습 필요.
* LangChain을 도입하여 PromptTemplate로 프롬프트의 재사용성과 모듈화를 높이는 방법을 학습할 예정.